In [0]:
# ============================================================
# DATA VALIDATION & QUALITY CHECKS - CORRECTED VERSION
# ============================================================

from pyspark.sql import functions as F
from datetime import datetime

# ============================================================
# GET ENVIRONMENT FROM ADF
# ============================================================

try:
    env = dbutils.widgets.get("environment")
    print(f"📌 Environment from ADF: {env}")
except:
    env = "DEV"
    print(f"📌 Using default: {env}")

# ============================================================
# STORAGE ACCOUNT
# ============================================================

storage_account_name = "capstonestorage01"

# ============================================================
# CONTAINER NAMES BASED ON ENVIRONMENT
# ============================================================

if env == 'DEV':
    silver_container = 'silver-dev'
    gold_container = 'gold-dev'
elif env == 'TEST':
    silver_container = 'silver-test'
    gold_container = 'gold-test'
else:  # PROD
    silver_container = 'silver'
    gold_container = 'gold'

# ============================================================
# BUILD PATHS
# ============================================================

SILVER = f"abfss://{silver_container}@{storage_account_name}.dfs.core.windows.net"
GOLD = f"abfss://{gold_container}@{storage_account_name}.dfs.core.windows.net"

print(f"\n{'='*55}")
print(f"🏭 ENVIRONMENT: {env}")
print(f"{'='*55}")
print(f"📁 SILVER Container: {silver_container}")
print(f"📁 GOLD Container: {gold_container}")
print(f"📍 SILVER Path: {SILVER}")
print(f"📍 GOLD Path: {GOLD}")
print(f"{'='*55}\n")

print("✅ Config ready!")

# ============================================================
# LOAD DATA FOR VALIDATION
# ============================================================

print("\n📖 Loading data for validation...")

# Load Silver layer data
df_transactions = spark.read.format("delta").load(f"{SILVER}/transactions")
df_customer = spark.read.format("delta").load(f"{SILVER}/customer_master")
df_account = spark.read.format("delta").load(f"{SILVER}/account_master")

print(f"✅ Transactions: {df_transactions.count():,} rows")
print(f"✅ Customers: {df_customer.count():,} rows")
print(f"✅ Accounts: {df_account.count():,} rows")

# ============================================================
# VALIDATION RULES
# ============================================================

print("\n" + "=" * 55)
print("  RUNNING DATA QUALITY CHECKS")
print("=" * 55)

validation_results = []
total_errors = 0

# Rule 1: amount >= 0
print("\n📋 Rule 1: amount >= 0")
invalid_amount = df_transactions.filter(F.col("amount") < 0)
invalid_count = invalid_amount.count()
if invalid_count > 0:
    print(f"  ❌ FAILED: {invalid_count} transactions with negative amount")
    total_errors += invalid_count
    invalid_amount.write.mode("overwrite").save(f"{GOLD}/rejects_negative_amount")
else:
    print("  ✅ PASSED: All amounts are valid")

# Rule 2: outstanding_balance >= 0
print("\n📋 Rule 2: outstanding_balance >= 0")
invalid_balance = df_transactions.filter(F.col("outstanding_balance") < 0)
invalid_count = invalid_balance.count()
if invalid_count > 0:
    print(f"  ❌ FAILED: {invalid_count} transactions with negative balance")
    total_errors += invalid_count
    invalid_balance.write.mode("overwrite").save(f"{GOLD}/rejects_negative_balance")
else:
    print("  ✅ PASSED: All balances are valid")

# Rule 3: credit_limit >= outstanding_balance
print("\n📋 Rule 3: credit_limit >= outstanding_balance")
df_joined = df_transactions.join(df_account, "account_id", "left")
invalid_limit = df_joined.filter(F.col("credit_limit") < F.col("outstanding_balance"))
invalid_count = invalid_limit.count()
if invalid_count > 0:
    print(f"  ❌ FAILED: {invalid_count} accounts where limit < outstanding")
    total_errors += invalid_count
    invalid_limit.write.mode("overwrite").save(f"{GOLD}/rejects_limit_exceeded")
else:
    print("  ✅ PASSED: All credit limits are sufficient")

# Rule 4: days_past_due >= 0 (if exists)
print("\n📋 Rule 4: days_past_due >= 0")
if "days_past_due" in df_transactions.columns:
    invalid_dpd = df_transactions.filter(F.col("days_past_due") < 0)
    invalid_count = invalid_dpd.count()
    if invalid_count > 0:
        print(f"  ❌ FAILED: {invalid_count} records with negative DPD")
        total_errors += invalid_count
    else:
        print("  ✅ PASSED: All DPD values are valid")
else:
    print("  ⚠️ SKIPPED: days_past_due column not found")

# Rule 5: transaction_date is not null
print("\n📋 Rule 5: transaction_date is not null")
null_date = df_transactions.filter(F.col("transaction_date").isNull())
invalid_count = null_date.count()
if invalid_count > 0:
    print(f"  ❌ FAILED: {invalid_count} transactions with null date")
    total_errors += invalid_count
else:
    print("  ✅ PASSED: All transaction dates are present")

# Rule 6: customer_id exists in customer master
print("\n📋 Rule 6: customer_id exists in customer master")
customer_ids = df_customer.select("customer_id").distinct()
orphan_transactions = df_transactions.join(
    customer_ids, "customer_id", "left_anti"
)
invalid_count = orphan_transactions.count()
if invalid_count > 0:
    print(f"  ❌ FAILED: {invalid_count} transactions with invalid customer_id")
    total_errors += invalid_count
    orphan_transactions.write.mode("overwrite").save(f"{GOLD}/rejects_orphan_customers")
else:
    print("  ✅ PASSED: All customer_ids are valid")

# Rule 7: account_id exists in account master
print("\n📋 Rule 7: account_id exists in account master")
account_ids = df_account.select("account_id").distinct()
orphan_accounts = df_transactions.join(
    account_ids, "account_id", "left_anti"
)
invalid_count = orphan_accounts.count()
if invalid_count > 0:
    print(f"  ❌ FAILED: {invalid_count} transactions with invalid account_id")
    total_errors += invalid_count
    orphan_accounts.write.mode("overwrite").save(f"{GOLD}/rejects_orphan_accounts")
else:
    print("  ✅ PASSED: All account_ids are valid")

# ============================================================
# VALIDATION SUMMARY
# ============================================================

print("\n" + "=" * 55)
print(f"  VALIDATION SUMMARY - {env}")
print("=" * 55)

if total_errors == 0:
    print("  ✅ ALL VALIDATION CHECKS PASSED!")
    print("  ✅ Data quality is GOOD!")
else:
    print(f"  ❌ VALIDATION FAILED! Found {total_errors} total errors")
    print("  ⚠️ Reject records saved to Gold layer")
    print("  ⚠️ Please review and fix data quality issues")

print(f"\n  Reject tables location: {GOLD}/rejects_*")
print("=" * 55)

# ============================================================
# RAISE ERROR IF VALIDATION FAILS (Optional - uncomment to fail pipeline)
# ============================================================

# if total_errors > 0:
#     raise Exception(f"Data validation failed with {total_errors} errors!")

In [0]:
df_txn   = spark.read.format("delta").load(f"{SILVER}/transactions")
df_deliq = spark.read.format("delta").load(f"{SILVER}/repayment_delinquency")
df_acc   = spark.read.format("delta").load(f"{SILVER}/account_master")
fact_txn = spark.read.format("delta").load(f"{GOLD}/fact_transactions")

print("✅ All tables loaded!")

In [0]:
print("=" * 60)
print("  BUSINESS RULE VALIDATION REPORT")
print("=" * 60)

results = []

# ── Rule 1: amount >= 0 ──
r1 = df_txn.filter(F.col("amount") < 0).count()
results.append(("Rule 1", "amount >= 0",             r1))

# ── Rule 2: outstanding_balance >= 0 ──
r2 = df_txn.filter(F.col("outstanding_balance") < 0).count()
results.append(("Rule 2", "outstanding_balance >= 0", r2))

# ── Rule 3: credit_limit >= outstanding_balance ──
r3 = fact_txn.filter(F.col("outstanding_balance") > F.col("credit_limit")).count()
results.append(("Rule 3", "credit_limit >= outstanding_balance", r3))

# ── Rule 4: days_past_due >= 0 ──
r4 = df_deliq.filter(F.col("days_past_due") < 0).count()
results.append(("Rule 4", "days_past_due >= 0",       r4))

# ── Rule 5: credit_limit > 0 ──
r5 = df_acc.filter(F.col("credit_limit") <= 0).count()
results.append(("Rule 5", "credit_limit > 0",         r5))

# ── Rule 6: no null transaction_id ──
r6 = df_txn.filter(F.col("transaction_id").isNull()).count()
results.append(("Rule 6", "transaction_id not null",  r6))

# ── Rule 7: no null customer_id ──
r7 = df_txn.filter(F.col("customer_id").isNull()).count()
results.append(("Rule 7", "customer_id not null",     r7))

# ── Rule 8: valid transaction_type ──
valid_types = ["Disbursement","Repayment","Fee","Interest"]
r8 = df_txn.filter(~F.col("transaction_type").isin(valid_types)).count()
results.append(("Rule 8", "valid transaction_type",   r8))

# ── Print Results ──
passed = 0
failed = 0
for rule_id, rule_desc, violations in results:
    status = "✅ PASS" if violations == 0 else "❌ FAIL"
    if violations == 0:
        passed += 1
    else:
        failed += 1
    print(f"  {status} | {rule_id} | {rule_desc:<40} | Violations: {violations:,}")

print("=" * 60)
print(f"  ✅ PASSED : {passed}")
print(f"  ❌ FAILED : {failed}")
print("=" * 60)

In [0]:
print("=" * 60)
print("  ROW COUNT RECONCILIATION")
print("=" * 60)

bronze_txn  = spark.read.format("delta").load(
    f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/transactions").count()
silver_txn  = df_txn.count()
gold_txn    = fact_txn.count()

print(f"  Bronze transactions  : {bronze_txn:>10,}")
print(f"  Silver transactions  : {silver_txn:>10,}  (removed {bronze_txn - silver_txn:,} dupes)")
print(f"  Gold fact_txn        : {gold_txn:>10,}  (should match silver)")

if silver_txn == gold_txn:
    print("\n  ✅ Silver → Gold row count MATCHES!")
else:
    print(f"\n  ⚠️  Mismatch! Difference = {abs(silver_txn - gold_txn):,} rows")

print("=" * 60)

In [0]:
# Capture all violations into a reject table for audit
df_rejects = df_txn.filter(
    (F.col("amount") < 0) |
    (F.col("outstanding_balance") < 0) |
    (F.col("transaction_id").isNull()) |
    (F.col("customer_id").isNull()) |
    (~F.col("transaction_type").isin(["Disbursement","Repayment","Fee","Interest"]))
).withColumn("reject_reason",
    F.when(F.col("amount") < 0,                F.lit("negative_amount"))
     .when(F.col("outstanding_balance") < 0,   F.lit("negative_balance"))
     .when(F.col("transaction_id").isNull(),    F.lit("null_transaction_id"))
     .when(F.col("customer_id").isNull(),       F.lit("null_customer_id"))
     .otherwise(F.lit("invalid_transaction_type"))
).withColumn("rejected_at", F.current_timestamp())

df_rejects.write.format("delta").mode("overwrite")\
    .option("overwriteSchema", "true")\
    .save(f"{GOLD}/reject_table")

print(f"✅ Reject table written : {df_rejects.count():,} records")
df_rejects.groupBy("reject_reason").count().show()

In [0]:
from datetime import datetime

audit_log = spark.createDataFrame([
    ("bronze", "transactions",          bronze_txn, "success", str(datetime.now())),
    ("silver", "transactions",          silver_txn, "success", str(datetime.now())),
    ("gold",   "fact_transactions",     gold_txn,   "success", str(datetime.now())),
    ("gold",   "fact_delinquency",
        spark.read.format("delta")
             .load(f"{GOLD}/fact_delinquency").count(), "success", str(datetime.now())),
    ("gold",   "dim_customer",
        spark.read.format("delta")
             .load(f"{GOLD}/dim_customer").count(),     "success", str(datetime.now())),
    ("gold",   "dim_account",
        spark.read.format("delta")
             .load(f"{GOLD}/dim_account").count(),      "success", str(datetime.now())),
],["layer","table_name","row_count","status","processed_at"])

audit_log.write.format("delta").mode("append")\
    .save(f"{GOLD}/audit_log")

print("✅ Audit log written!")
audit_log.show(truncate=False)

In [0]:
print("=" * 60)
print("  VALIDATION COMPLETE — PIPELINE READY!")
print("=" * 60)
print(f"  ✅ Business Rules Passed : {passed}")
print(f"  ❌ Business Rules Failed : {failed}")
print(f"  📋 Reject Records        : {df_rejects.count():,}")
print(f"  📋 Audit Log Entries     : {audit_log.count()}")
print("=" * 60)
print("  🚀 READY FOR AIRFLOW DAG + ADF PIPELINE!")
print("=" * 60)

In [0]:
print("=" * 60)
print("  WHICH RULE FAILED?")
print("=" * 60)
for rule_id, rule_desc, violations in results:
    if violations > 0:
        print(f"  ❌ {rule_id} | {rule_desc} | Violations: {violations:,}")

# Show sample violating records
print("\n  Sample records violating Rule 3:")
fact_txn.filter(F.col("outstanding_balance") > F.col("credit_limit"))\
    .select("transaction_id","account_id","outstanding_balance","credit_limit")\
    .show(10)